In [1]:
#kernel thesis clean4
import pickle
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_pickle("C:\\Users\\Patrick\\Masterthesis\\Benchmarks\\screw_data_s02-v2_identical-to-v1.pkl")
df.head()

,time_values,torque_values,angle_values,gradient_values,step_values,class_values,workpiece_location,workpiece_usage,workpiece_result,scenario_condition,scenario_exception
0,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.067, 0.077, 0.126, 0.087, 0.089, 0.097, 0.0...","[0.5, 1.25, 2.25, 3.75, 5.0, 6.25, 7.5, 8.75, ...","[0.0, 0.0, 0.0298, 0.0214, 0.0064, 0.0023, -0....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,0,OK,normal,0
1,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.005, 0.008, 0.061, 0.069, 0.104, 0.124, 0....","[0.0, 0.25, 0.75, 1.5, 2.5, 4.0, 5.25, 6.5, 7....","[0.0, 0.0, 0.0, 0.0, 0.0287, 0.0282, 0.0091, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,0,OK,normal,0
2,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, 0.01, 0.039, 0.099, 0.102, 0.087, 0.0...","[0.0, 0.5, 1.25, 2.25, 3.5, 4.75, 6.0, 7.5, 8....","[0.0, 0.0, 0.0, 0.0196, 0.0223, 0.0211, 0.0096...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,1,OK,normal,0
3,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.081, 0.059, 0.12, 0.077, 0.043, 0.059, 0.06...","[0.75, 1.75, 2.75, 4.0, 5.25, 6.5, 7.75, 9.25,...","[0.0, 0.0, 0.0261, 0.0207, 0.0, -0.0036, -0.00...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,1,OK,normal,0
4,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, -0.0012, 0.0006, 0.0024, 0.0042, 0.00...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.5, 2.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.025...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,2,OK,normal,0


# PINN Feature Formel: Gradient nach T in abhängigkeit zu angle
## Quelle: Eigene interpretation

In [6]:
torque = np.array(df['torque_values'].tolist())
angle  = np.array(df['angle_values'].tolist())

print("torque shape:", torque.shape)
print("angle shape:", angle.shape)

angle_rad = np.radians(angle)
print("angle_rad shape:", angle_rad.shape)

d_Torque = np.gradient(torque, axis=1)
print("d_Torque shape:", np.array(d_Torque).shape)

d_angle = np.gradient(angle_rad, axis=1)
print("d_angle shape:", np.array(d_angle).shape)

d_T_a = d_Torque / (d_angle + 1e-8)
print("d_T_a shape:", np.array(d_T_a).shape)

torque shape: (12500, 800)
angle shape: (12500, 800)
angle_rad shape: (12500, 800)
d_Torque shape: (12500, 800)
d_angle shape: (12500, 800)
d_T_a shape: (12500, 800)


In [7]:
x_data = np.array(df['torque_values'].tolist())[..., np.newaxis]  
d_T_a = np.array(d_T_a)[..., np.newaxis]
x_data = np.concatenate([x_data, d_T_a], axis=-1)  #concat auf letzter achse
y_data = np.array(df['class_values'].tolist())
le = LabelEncoder()
y_encoded = le.fit_transform(y_data)

x_data = np.transpose(x_data, (0, 2, 1))  #reshaped to (num_samples, num_features, sequence_length)

X_train_full, X_test, y_train_full, y_test = train_test_split(x_data, y_encoded, test_size=0.2, stratify=y_encoded,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full,y_train_full,test_size=0.25,stratify=y_train_full,random_state=42) #0.25 * 0.8 = 0.2

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
val_dataset   = TensorDataset(X_val, y_val)
test_dataset  = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: torch.Size([7500, 2, 800]) torch.Size([7500])
Validation: torch.Size([2500, 2, 800]) torch.Size([2500])
Test: torch.Size([2500, 2, 800]) torch.Size([2500])


In [8]:
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [ ]:
import torch
import torch.nn as nn
import sys
sys.path.append(r"C:\Users\Patrick\InceptionTime-Pytorch")
from inception import InceptionBlock


class Flatten(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x.mean(-1)   #safer als view


model = nn.Sequential(
    InceptionBlock(
        in_channels=2,  # 2 Features: torque and E_kin
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    InceptionBlock(
        in_channels=32 * 4,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    nn.AdaptiveAvgPool1d(1),
    Flatten(),
    nn.Linear(32 * 4, 8)   # 8 Klassen
)

In [10]:
from sklearn.metrics import f1_score
import math
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

earlystop = EarlyStopper(patience=20, min_delta=0.001)

epochs = 100

train_losses = []
val_losses = []
val_f1_scores = []

best_val_f1 = -np.inf
best_model_state = None

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    model.eval()

    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_val_batch, y_val_batch in val_loader:
            X_val_batch = X_val_batch.to(device)
            y_val_batch = y_val_batch.to(device)

            outputs = model(X_val_batch)
            loss = criterion(outputs, y_val_batch)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_val_batch.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    val_f1 = f1_score(all_labels, all_preds, average="macro")
    val_f1_scores.append(val_f1)
    scheduler.step(avg_val_loss)

    #best model speichern für test
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict()

    if earlystop.early_stop(avg_val_loss):
        print("Early stopping triggered")
        break

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val F1 Score: {val_f1:.4f}")


Epoch 1/100, Train Loss: 1.9840, Val Loss: 1.9104, Val F1 Score: 0.1209
Epoch 2/100, Train Loss: 1.8544, Val Loss: 2.1298, Val F1 Score: 0.1230
Epoch 3/100, Train Loss: 1.7892, Val Loss: 1.7693, Val F1 Score: 0.2267
Epoch 4/100, Train Loss: 1.7571, Val Loss: 4.3140, Val F1 Score: 0.0227
Epoch 5/100, Train Loss: 1.7348, Val Loss: 5.3712, Val F1 Score: 0.0227
Epoch 6/100, Train Loss: 1.7187, Val Loss: 5.4246, Val F1 Score: 0.0272
Epoch 7/100, Train Loss: 1.7070, Val Loss: 2.2773, Val F1 Score: 0.0859
Epoch 8/100, Train Loss: 1.6813, Val Loss: 1.7486, Val F1 Score: 0.2481
Epoch 9/100, Train Loss: 1.6676, Val Loss: 3.0610, Val F1 Score: 0.0447
Epoch 10/100, Train Loss: 1.6584, Val Loss: 6.2847, Val F1 Score: 0.0956
Epoch 11/100, Train Loss: 1.6506, Val Loss: 16.7009, Val F1 Score: 0.0227
Epoch 12/100, Train Loss: 1.6487, Val Loss: 1.7735, Val F1 Score: 0.2302
Epoch 13/100, Train Loss: 1.6239, Val Loss: 4.4987, Val F1 Score: 0.0248
Epoch 14/100, Train Loss: 1.6250, Val Loss: 1.6764, Val F1 

In [11]:
model.load_state_dict(best_model_state)
#torch.save(best_model_state, "best_inception_model.pth")
model.eval()

test_preds = []
test_labels = []
test_loss = 0.0

with torch.no_grad():
    for X_test_batch, y_test_batch in test_loader:
        X_test_batch = X_test_batch.to(device)
        y_test_batch = y_test_batch.to(device)

        outputs = model(X_test_batch)
        loss = criterion(outputs, y_test_batch)

        test_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(y_test_batch.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_f1 = f1_score(test_labels, test_preds, average="macro")

print(f"Test Loss: {test_loss:.4f}")
print(f"Test F1 Macro: {test_f1:.4f}")

Test Loss: 1.6856
Test F1 Macro: 0.2739
